# TD 10 — Révision : le sujet d'annale, vérifié par le calcul

**Analyse des données — L3 Économie**

L'épreuve se passe **sur papier, sans machine** : ce notebook ne sert pas à y répondre, il sert à **vérifier** ce qu'on a répondu. Chaque réponse du corrigé y est recalculée.

Travaillez dans cet ordre : faites le sujet au stylo, puis ouvrez ce notebook et exécutez-le section par section. Quand un chiffre ne tombe pas comme le vôtre, c'est là qu'il y a quelque chose à comprendre.

Les données sont **fictives** — construites pour que chaque piège du semestre apparaisse exactement une fois. Les mécanismes, eux, sont ceux que vous rencontrerez.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

URL = ("https://raw.githubusercontent.com/StefaniaMarcassa/"
       "analyse_des_donnees/main/data/td10-loyers.csv")

df = pd.read_csv(URL, sep=";", dtype={"IDENT": str, "REGION": str})
print(df.shape)
df.head(3)

Deux arguments décident déjà de tout : `sep=";"` (le fichier est séparé par des points-virgules) et `dtype={"IDENT": str, "REGION": str}`. Sans le second, `IDENT` perdrait ses zéros initiaux et `REGION` deviendrait un nombre — c'est la **question 6**.

---
## Partie 1 — Lire le fichier

**Q1–Q2 : que décrit une ligne, et quelle est la clé ?**

In [ ]:
print("lignes          :", len(df))
print("IDENT distincts :", df["IDENT"].nunique())
print("doublons IDENT  :", len(df) - df["IDENT"].nunique())

# la cle est le couple IDENT + NOLOG : il ne se repete jamais
print("couples IDENT+NOLOG distincts :", len(df[["IDENT", "NOLOG"]].drop_duplicates()))

df[df["IDENT"] == "00007"]

2 560 lignes pour 2 400 identifiants : **une ligne est un logement, pas un ménage**. 160 ménages en occupent deux. Le couple `IDENT` + `NOLOG`, lui, ne se répète jamais.

**Q3 : combien de valeurs manquantes ?**

In [ ]:
print("isna() sur tout le fichier :", df.isna().sum().sum())

for col, code in [("LOYER", 9999), ("SURFACE", 999),
                  ("REVENU", 999999), ("ANCIENNETE", -1)]:
    print(f"{col:<11} == {code:<7} : {(df[col] == code).sum()}")
print("LOYER       == 0       :", (df["LOYER"] == 0).sum())

`isna()` renvoie **zéro**, et pourtant 775 valeurs sont absentes. Elles sont codées par des **nombres** : aucune case n'est vide, Python lit 9 999 comme un loyer valide. Elles ne deviendront manquantes qu'après un `replace` explicite.

**Q4 : `LOYER = 0` et `ANCIENNETE = -1` — erreur, non-réponse, ou autre chose ?**

In [ ]:
# les deux codes se repondent : ce sont les memes lignes
print("LOYER == 0 et ANCIENNETE == -1 :",
      ((df["LOYER"] == 0) & (df["ANCIENNETE"] == -1)).sum())

df[df["LOYER"] == 0].head(3)

Les deux codes désignent exactement les mêmes 390 lignes : ce sont les **propriétaires**. Ni erreur, ni non-réponse — un **hors champ** : la question ne leur a pas été posée. On les **écarte du champ**, on ne les corrige pas.

**Q5 : le ménage 00007 apparaît deux fois avec le même REVENU.**

In [ ]:
print("somme REVENU sur toutes les lignes :", df["REVENU"].sum())
print("somme REVENU par menage            :",
      df.drop_duplicates("IDENT")["REVENU"].sum())

`REVENU` et `POIDS` sont des caractéristiques du **ménage**, donc répétées sur chacune de ses lignes. Sommer sur tout le fichier compte deux fois le revenu des 160 ménages à deux logements. Le bon calcul passe par `drop_duplicates("IDENT")`.

**Q7 : reconstituer la chaîne de nettoyage.**

In [ ]:
etapes = {"fichier brut": df["LOYER"].mean()}

d = df.copy()
d["LOYER"]   = d["LOYER"].replace(9999, np.nan)
etapes["9999 -> NaN"] = d["LOYER"].mean()

d = d[d["LOYER"] > 0]                     # locataires seulement
etapes["locataires"] = d["LOYER"].mean()

d = d[d["SURFACE"] != 999].copy()         # surface renseignee
etapes["surface renseignee"] = d["LOYER"].mean()

for k, v in etapes.items():
    print(f"{k:<20} {v:>9.2f} EUR")
print("\nobservations retenues :", len(d))

La première étape fait **chuter** la moyenne : 195 loyers à 9 999 € pesaient près de deux millions d'euros. La deuxième la fait **remonter** : on retire 390 zéros. **1 620,63 € n'était le loyer de personne.**

---
## Partie 2 — Diagnostiquer du code

Les quatre défaillances de cette partie ont un point commun : **aucune ne lève d'erreur**. Le code tourne, et le résultat est faux.

**Q8 : l'affectation sur une copie.**

In [ ]:
test = df.copy()
test[test["LOYER"] == 9999]["LOYER"] = np.nan     # ne fait rien
print("valeurs manquantes apres :", test["LOYER"].isna().sum())

test["LOYER"] = test["LOYER"].replace(9999, np.nan)   # la forme correcte
print("avec .replace             :", test["LOYER"].isna().sum())

L'affectation porte sur un **extrait temporaire**, aussitôt jeté. Selon la version de pandas vous verrez un avertissement, ou rien du tout — mais jamais une erreur.

**Q9 : `NaN > 0` vaut `False`.**

In [ ]:
t = df.copy()
t["LOYER"] = t["LOYER"].replace(9999, np.nan)

print("lignes au depart :", len(t))
print("apres LOYER > 0  :", len(t[t["LOYER"] > 0]))
print("ecartees         :", len(t) - len(t[t["LOYER"] > 0]))
print("  dont proprietaires :", (t["LOYER"] == 0).sum())
print("  dont non-reponses  :", t["LOYER"].isna().sum())

585 lignes écartées : les 390 propriétaires **et** les 195 non-réponses. Le résultat est celui qu'il fallait, mais pas pour la raison qu'on croyait. Un filtre qui fait plus que prévu reste un filtre à expliciter.

**Q10–Q11 : la fusion qui gonfle, et la comparaison qui ne compare rien.**

In [ ]:
# Q10 : une table de droite qui contient un doublon de cle
regions = pd.DataFrame({"REGION": ["11", "32", "53", "76", "84", "11"],
                        "nom": ["Ile-de-France", "Hauts-de-France", "Bretagne",
                                "Occitanie", "Auvergne-Rhone-Alpes", "IDF (doublon)"]})

print("apres merge how='left' :", len(df.merge(regions, how="left", on="REGION")))
try:
    df.merge(regions, how="left", on="REGION", validate="m:1")
except Exception as e:
    print("avec validate='m:1'    :", type(e).__name__, "->", e)

# Q11 : REGION est du texte
print("\ndf[df['REGION'] == 11]  ->", len(df[df["REGION"] == 11]),  "lignes")
print("df[df['REGION'] == '11'] ->", len(df[df["REGION"] == "11"]), "lignes")

`validate="m:1"` lève une erreur au lieu de gonfler silencieusement le tableau. Et `"11" == 11` est faux : la comparaison renvoie `False` partout, d'où un tableau **vide, sans aucun message**.

**Q12 : compter des lignes n'est pas compter des ménages.**

In [ ]:
pd.DataFrame({"logements": df.groupby("REGION")["IDENT"].count(),
              "menages"  : df.groupby("REGION")["IDENT"].nunique()})

---
## Partie 3 — Décrire, pondérer, repérer

**Q13–Q14 : l'extrait de 12 lignes.**

In [ ]:
# les 12 lignes de l'annexe du dossier
CLES = [("00005", 1), ("00006", 1), ("00007", 1), ("00007", 2), ("00037", 1),
        ("00049", 1), ("00104", 1), ("00378", 1), ("00956", 1), ("01201", 1),
        ("01994", 1), ("02073", 1)]

e = df.set_index(["IDENT", "NOLOG"]).loc[CLES].reset_index()
e["LOYER"]   = e["LOYER"].replace(9999, np.nan)
e["SURFACE"] = e["SURFACE"].replace(999, np.nan)
e["REVENU"]  = e["REVENU"].replace(999999, np.nan)

print("valeurs reellement manquantes :", e.isna().sum().sum())
print("moyenne brute des 12 lignes   :", round(df.set_index(["IDENT","NOLOG"])
                                               .loc[CLES, "LOYER"].mean(), 2))

loc = e[(e["LOYER"] > 0) & e["SURFACE"].notna()]
print("\nlignes retenues  :", len(loc))
print("moyenne          :", round(loc["LOYER"].mean(), 2))
print("mediane          :", round(loc["LOYER"].median(), 2))
print("moyenne ponderee :",
      round(np.average(loc["LOYER"], weights=loc["POIDS"]), 2))

La moyenne pondérée est **supérieure** à la moyenne simple : les logements aux loyers élevés portent des poids plus grands, donc ils sont **sous-représentés dans l'échantillon** par rapport à la population. La pondération corrige ce déséquilibre — elle ne réduit pas l'incertitude.

**Q15 : écart-type contre erreur-type.**

In [ ]:
ecart  = d["LOYER"].std()
erreur = ecart / np.sqrt(len(d))

print("moyenne     :", round(d["LOYER"].mean(), 2))
print("ecart-type  :", round(ecart, 1), "  <- dispersion entre logements")
print("erreur-type :", round(erreur, 1), "  <- incertitude sur la moyenne")
print("IC 95 %     : [", round(d["LOYER"].mean() - 1.96*erreur, 0), ",",
                          round(d["LOYER"].mean() + 1.96*erreur, 0), "]")

L'écart-type **ne diminue pas** quand l'échantillon grandit ; l'erreur-type, si. C'est la seconde qui donne l'intervalle de confiance.

**Q16–Q17 : la règle des 1,5 écarts interquartiles.**

In [ ]:
q1, q3 = d["LOYER"].quantile([.25, .75])
iqr = q3 - q1
bas, haut = q1 - 1.5*iqr, q3 + 1.5*iqr

print(f"q1 = {q1:.0f}   q3 = {q3:.0f}   IQR = {iqr:.0f}")
print(f"seuils : bas = {bas:.0f}   haut = {haut:.0f}")
print("signales en haut :", (d["LOYER"] > haut).sum())
print("signales en bas  :", (d["LOYER"] < bas).sum())

# le test a faire AVANT de decider : avec et sans
print("\nmoyenne avec    :", round(d["LOYER"].mean(), 2))
print("moyenne sans    :", round(d.loc[d["LOYER"] <= haut, "LOYER"].mean(), 2))

Le seuil bas est **négatif** : il ne peut désigner personne. La règle est symétrique, la distribution ne l'est pas — elle est aveugle d'un côté.

Faut-il retirer les loyers signalés en haut ? **Non** : un loyer de 2 500 € en Île-de-France est un loyer ordinaire. Le test à faire avant de décider est celui qu'on vient d'exécuter : recalculer **avec et sans**, et publier l'écart.

---
## Partie 4 — Régresser

C'est le cœur du sujet. On construit les trois colonnes du dossier.

In [ ]:
d["lloyer"] = np.log(d["LOYER"])
d["lsurf"]  = np.log(d["SURFACE"])
d["maison"] = (d["TYPE"] == 2).astype(int)

f = ["lloyer ~ lsurf",
     "lloyer ~ lsurf + PIECES + maison",
     "lloyer ~ lsurf + PIECES + maison + C(REGION)"]

modeles = [smf.ols(x, data=d).fit(cov_type="HC1") for x in f]

tableau = pd.DataFrame({
    f"({i+1})": [round(m.params.get(v, np.nan), 3) for v in
                 ["lsurf", "PIECES", "maison"]] + [round(m.rsquared, 3)]
    for i, m in enumerate(modeles)},
    index=["lsurf", "PIECES", "maison", "R2"])
tableau

In [ ]:
# les ecarts-types, qui sont la reponse a la question 21
pd.DataFrame({f"({i+1})": [round(m.bse.get(v, np.nan), 4) for v in
                           ["lsurf", "PIECES", "maison"]]
              for i, m in enumerate(modeles)},
             index=["lsurf", "PIECES", "maison"])

**Q21 : ce qui a changé pour `PIECES`, c'est l'écart-type, pas le coefficient.** Il passe de 0,009 à 0,006 : les effets fixes absorbent une grande partie de la variance résiduelle, donc l'estimation devient plus **précise**. Une variable peut devenir significative sans que son effet estimé ait bougé.

**Q22 : le coefficient de `maison` change de signe. Pourquoi ?**

In [ ]:
for i, m in enumerate(modeles[1:], start=2):
    c = m.params["maison"]
    print(f"colonne ({i}) : {c:+.3f}  ->  {100*(np.exp(c)-1):+.1f} % de loyer")

g = pd.DataFrame({
    "part de maisons": d.groupby("REGION")["maison"].mean(),
    "loyer au m2"    : (d["LOYER"] / d["SURFACE"]).groupby(d["REGION"]).mean()})
print()
print(g.round(3))
print("\ncorrelation :", round(g.corr().iloc[0, 1], 3))

94 % de maisons dans la région la moins chère, 24 % dans la plus chère : la corrélation vaut **−0,99**.

Sans effets fixes, la muette `maison` ne compare donc pas une maison à un appartement — **elle compare la province à l'Île-de-France**. Avec eux, la comparaison se fait *à l'intérieur de chaque région*, et le signe s'inverse. **Le coefficient ne s'est pas corrigé : il a changé de question.**

**Q24 : ce que les effets fixes rendent inestimable.** Ajoutons une variable constante dans chaque région — le revenu médian régional, par exemple.

In [ ]:
rev_reg = d.groupby("REGION")["REVENU"].median()
d["rev_region"] = d["REGION"].map(rev_reg)

m = smf.ols("lloyer ~ lsurf + maison + rev_region + C(REGION)", data=d).fit()

print("colonnes de la matrice :", m.model.exog.shape[1])
print("rang de la matrice     :", np.linalg.matrix_rank(m.model.exog))

# le logiciel renvoie quand meme des nombres : on compare les effets de region
pd.DataFrame({"sans rev_region": modeles[2].params.filter(like="REGION"),
              "avec rev_region": m.params.filter(like="REGION")}).round(3)

La matrice a **huit colonnes pour un rang de sept** : `rev_region` est une combinaison exacte des muettes de région. Le logiciel ne s'arrête pas pour autant — il renvoie des nombres, et ces nombres **ne sont pas identifiés** : les effets de région se déplacent de −0,858 à −1,044 selon la façon dont le calcul répartit l'effet entre les colonnes redondantes.

On renonce donc à expliquer *pourquoi une région est chère* pour mieux estimer ce qui fait varier les loyers *en son sein*. **Un effet fixe ne corrige rien : il absorbe**, et ce qu'il absorbe est perdu.

---
## Partie 5 — Prévoir

**Q26–Q27 : l'arbre du dossier, et les cinq modèles.**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor

X = pd.get_dummies(d[["lsurf", "PIECES", "maison", "ANCIENNETE", "REGION"]],
                   columns=["REGION"], prefix="reg", drop_first=True).astype(float)
y = d["lloyer"]
Xa, Xt, ya, yt = train_test_split(X, y, test_size=0.2, random_state=0)
print("apprentissage", len(Xa), "| test", len(Xt))

mod = {"regression"     : LinearRegression(),
       "arbre prof. 2"  : DecisionTreeRegressor(max_depth=2,  random_state=0),
       "arbre prof. 4"  : DecisionTreeRegressor(max_depth=4,  random_state=0),
       "arbre prof. 20" : DecisionTreeRegressor(max_depth=20, random_state=0),
       "foret"          : RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                                   random_state=0)}

def r2_hors(vrai, prev, repere):
    vrai = np.asarray(vrai)
    return 1 - np.sum((vrai - prev)**2) / np.sum((vrai - repere)**2)

lignes = {}
for n, m in mod.items():
    m.fit(Xa, ya)
    lignes[n] = [round(m.score(Xa, ya), 3),
                 round(r2_hors(yt, m.predict(Xt), ya.mean()), 3)]

pd.DataFrame(lignes, index=["apprentissage", "hors echantillon"]).T

In [ ]:
a = DecisionTreeRegressor(max_depth=2, random_state=0).fit(Xa, ya)

fig, ax = plt.subplots(figsize=(11, 5))
plot_tree(a, feature_names=X.columns, filled=True, rounded=True,
          impurity=False, fontsize=9, ax=ax)
plt.show()

# la question 26, en passant par le chemin
cas = pd.DataFrame([dict.fromkeys(X.columns, 0.0)] * 2, index=["HdF", "IdF"])
cas["lsurf"] = np.log(95)
cas.loc["HdF", "reg_32"] = 1.0
print("\nprevision pour 95 m2 :")
print(np.exp(a.predict(cas[X.columns])).round(0))

$\log(95) = 4{,}55$ : on part **à droite** à la racine. En Hauts-de-France on repart à droite, en Île-de-France à gauche — d'où deux prévisions très différentes pour le **même logement**.

Remarquez que l'arbre prédit **la même valeur** pour tous les logements d'une feuille : 95 ou 140 m² donnent le même loyer. C'est le prix des paliers.

**Q29 : pourquoi la forêt perd-elle contre la régression ?**

In [ ]:
# la relation est-elle lineaire ? on la regarde.
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.scatter(d["lsurf"], d["lloyer"], s=4, alpha=.15, color="#9aa3a0", linewidths=0)
xx = np.linspace(d["lsurf"].min(), d["lsurf"].max(), 100)
m1 = smf.ols("lloyer ~ lsurf", data=d).fit()
ax.plot(xx, m1.params["Intercept"] + m1.params["lsurf"]*xx, color="#E2162F", lw=2)
ax.set_xlabel("log(surface)"); ax.set_ylabel("log(loyer)")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

La relation est **effectivement linéaire**, et les effets de région sont additifs : la régression épouse exactement la forme des données. La souplesse de la forêt ne sert alors à rien, et elle coûte — elle approche une droite par des paliers.

**Un modèle plus puissant n'est pas un modèle meilleur.** Il l'est seulement quand la forme vraie lui est inaccessible autrement.

**Q30 : une importance n'est pas un coefficient.**

In [ ]:
foret = mod["foret"]
imp = pd.Series(foret.feature_importances_, index=X.columns).sort_values(ascending=False)
print(imp.round(3).head(5))

print("\nce que l'importance NE dit pas : ni le signe, ni l'unite, ni l'incertitude.")
print("le coefficient, lui, le dit :",
      round(modeles[2].params["lsurf"], 3), "log-point par log-point,")
print("soit environ +0,85 % de loyer pour +1 % de surface.")

---
## Q31 — Rédiger

Le dernier point du sujet ne se calcule pas. Il demande cinq lignes défendables :

> Sur 1 884 logements loués, à surface, nombre de pièces et région donnés, une surface supérieure de 1 % est associée à un loyer supérieur de 0,85 %, et une maison se loue 4,8 % de plus qu'un appartement comparable. Les écarts entre régions sont considérables : à logement identique, le loyer est inférieur de 86 log-points dans les Hauts-de-France par rapport à l'Île-de-France. Ces coefficients décrivent des associations sur données d'enquête en coupe ; ils ne mesurent pas l'effet causal d'un agrandissement, et le modèle n'observe ni l'état du logement, ni l'étage, ni la proximité des transports.

Trois choses y sont exigées : les coefficients **traduits en pourcentage**, la mention explicite de **ce qui est tenu constant**, et une limite **nommée**. Une copie qui écrit « une maison *fait* gagner 4,8 % » perd le point.